In [11]:
import pandas as pd
import numpy as np
from ruptures import Pelt

clustered_events = pd.read_csv('clustered_rainfall_events.csv')
original_df = pd.read_csv('Data/pakistan_rain.csv', skiprows=[1])

original_df['date'] = pd.to_datetime(original_df['date'])
original_df = original_df.sort_values(['adm2_id', 'date'])

clustered_events = clustered_events.sort_values(['district_id', 'event_index'])
clustered_events['district'] = clustered_events['district_id']
clustered_events['typology'] = clustered_events['rainfall_type_name'].astype(str)

print("Recovering real year for each event from original data...")

year_map = {}
for district in original_df['adm2_id'].unique():
    dist_df = original_df[original_df['adm2_id'] == district].copy()
    dist_df['is_wet'] = dist_df['rfh'] > 5
    dist_df['event_start'] = dist_df['is_wet'] & ~dist_df['is_wet'].shift(1, fill_value=False)
    dist_df['event_index'] = dist_df['event_start'].cumsum() - 1
    
    grouped = dist_df[dist_df['is_wet']].groupby('event_index')
    for e_idx, group in grouped:
        rep_year = int(group['date'].dt.year.mode()[0])
        year_map[(district, int(e_idx))] = rep_year

clustered_events['year'] = clustered_events.apply(
    lambda row: year_map.get((row['district'], row['event_index']), 2000), axis=1
)

print("Year recovery complete. Sample:")
print(clustered_events[['district', 'event_index', 'year', 'typology']].head(6))

def calculate_yearly_proportions(events_df):
    grouped = events_df.groupby(['district', 'year', 'typology']).size().unstack(fill_value=0)
    proportions = grouped.div(grouped.sum(axis=1), axis=0).reset_index()
    return proportions

def detect_regime_shifts(proportions_df, pen=8, min_years=8):
    shifts = []
    print("\nRunning PELT on real yearly proportions...")
    
    for district in sorted(proportions_df['district'].unique()):
        df_d = proportions_df[proportions_df['district'] == district].sort_values('year')
        
        if len(df_d) < min_years:
            continue
            
        signal = df_d['Type 0'].values
        
        try:
            model = Pelt(model="rbf").fit(signal)
            cps = model.predict(pen=pen)
            
            if len(cps) > 1:
                cp_idx = cps[0]
                shift_year = int(df_d['year'].iloc[cp_idx])
                
                shifts.append({
                    'district': district,
                    'shift_year': shift_year,
                    'total_years': len(df_d),
                    'confidence': 'high'
                })
                print(f"District {district} → Regime shift detected in {shift_year}")
            else:
                print(f"District {district} → No shift")
        except:
            pass
    
    shift_df = pd.DataFrame(shifts)
    if shift_df.empty:
        print("No shifts detected. Try lowering pen=5")
    
    return shift_df

yearly_props_df = calculate_yearly_proportions(clustered_events)
shift_df = detect_regime_shifts(yearly_props_df, pen=8)

shift_df.to_csv('detected_regime_shifts.csv', index=False)
yearly_props_df.to_csv('yearly_typology_proportions.csv', index=False)

print("\n=== FINAL RESULTS ===")
print(shift_df)

Recovering real year for each event from original data...
Year recovery complete. Sample:
   district  event_index  year typology
0   1009006            0  1981   Type 4
1   1009006            1  1982   Type 4
2   1009006            2  1983   Type 0
3   1009006            3  1983   Type 0
4   1009006            4  1983   Type 0
5   1009006            5  1983   Type 4

Running PELT on real yearly proportions...
District 1009006 → No shift
District 1009007 → No shift
District 1009008 → No shift
District 1009009 → No shift
District 1009010 → No shift
District 1009011 → No shift
District 1009012 → No shift
District 1009013 → No shift
District 1009014 → No shift
District 1009015 → No shift
District 1009016 → No shift
District 1009017 → No shift
District 1009018 → No shift
District 1009019 → No shift
District 1009020 → No shift
District 1009021 → No shift
District 1009022 → No shift
District 1009023 → No shift
District 1009024 → No shift
District 1009025 → No shift
District 1009026 → No shif